# Backtest de la Magic Formula en las grandes empresas de EE. UU. (2015–2026)

**Pregunta:** si el 1 de enero de cada año desde 2015 hubiera comprado a partes iguales las 30 primeras empresas del S&P 500 según la Magic Formula de Greenblatt, y las hubiera mantenido un año, ¿cuánto habría ganado frente al índice?

**Datos (WRDS, suscripción de CUNEF):**
- **CRSP**: rentabilidades mensuales con dividendos (incluidas las de las empresas que dejaron de cotizar) y la capitalización de cada acción.
- **Compustat (fundamentales trimestrales)**: EBIT, balance y la fecha en que se publicó cada dato.
- **CRSP/Compustat Merged**: el enlace entre las dos bases.

**Cómo evito los dos sesgos clásicos:**
1. *Sesgo de supervivencia*: cada año uso las empresas que **estaban** en el universo ese día (S&P 500 histórico o las 500 mayores), no las de hoy. Si una empresa deja de cotizar, CRSP incluye la rentabilidad de su salida y el dinero pasa a liquidez.
2. *Sesgo de anticipación (look-ahead)*: solo uso cuentas **ya publicadas** antes de cada fecha (campo `rdq` de Compustat).

**Cómo usar este notebook:** ejecuta las celdas en orden con `Shift + Enter`. Si alguna da error, copia el mensaje completo.

> Los datos de WRDS son para uso académico y no se pueden redistribuir. Las tablas que se guardan al final contienen solo **resultados agregados** del estudio.

## 1. Parámetros del estudio
Aquí se cambian las opciones sin tocar el resto del código.

In [ ]:
ANIOS = list(range(2015, 2027))   # 1 de enero de cada año. 2026 es el año en curso (parcial)
TOP_N = 30                        # nº de empresas de la cartera (Greenblatt: 20-30)
ROC_METODO = "tangible"           # "tangible" = Greenblatt; "gw" = sumando fondo de comercio e intangibles
UNIVERSO = "auto"                 # "auto": S&P 500 histórico solo si está completo; "mayores": las N mayores
TAMANO_UNIVERSO = 500             # tamaño del universo de las mayores empresas
USUARIO_WRDS = "jaimeandres"

INICIO_DATOS = "2012-01-01"       # margen para tener 4 trimestres de EBIT en 2015
FIN_DATOS = "2026-12-31"

## 2. Conexión con WRDS
Te pedirá la contraseña (escríbela tú) y puede que tengas que aprobar el aviso de **Duo** en el móvil.

In [ ]:
import wrds
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 20)
import warnings
# Python 3.14 confunde el detector de "chained assignment" de pandas y da falsas alarmas
warnings.filterwarnings("ignore", message=".*[Cc]hained assignment.*")
warnings.filterwarnings("ignore", category=FutureWarning)

db = wrds.Connection(wrds_username=USUARIO_WRDS)

## 3. Localizar las tablas
CRSP tiene dos formatos: el antiguo (`msf`) y el nuevo de 2024 (`msf_v2`). Esta celda detecta cuál tiene CUNEF y qué columnas usa, y descarga el **enlace entre CRSP y Compustat** (qué acción de CRSP corresponde a cada empresa de Compustat).

In [ ]:
def columnas(lib, tabla):
    return set(db.describe_table(library=lib, table=tabla)["name"].str.lower())

def elegir(cols, *opciones):
    for o in opciones:
        if o in cols:
            return o
    raise KeyError(f"No encuentro ninguna de {opciones}. Columnas disponibles: {sorted(cols)}")

def sql_lista(valores):
    # Lista para una consulta SQL: (10104, 10107) o ('001004', '001045'), sin tipos de numpy
    limpios = sorted({int(v) if isinstance(v, (int, np.integer)) else str(v) for v in valores})
    return "(" + ", ".join(str(v) if isinstance(v, int) else "'" + v.replace("'", "''") + "'" for v in limpios) + ")"

def intentar(sql, date_cols=None, aviso=""):
    # Ejecuta una consulta; si no hay permiso o no existe la tabla, devuelve None en vez de parar
    try:
        return db.raw_sql(sql, date_cols=date_cols)
    except Exception as e:
        print(f"   · {aviso}: no disponible ({str(e).splitlines()[0][:100]})")
        return None

tablas_crsp = set(db.list_tables(library="crsp"))

# Rentabilidades mensuales
if "msf_v2" in tablas_crsp:
    MSF = "msf_v2"
    c = columnas("crsp", MSF)
    COL_FECHA, COL_RET = elegir(c, "mthcaldt", "date"), elegir(c, "mthret", "ret")
    COL_PRC, COL_SHR = elegir(c, "mthprc", "prc"), elegir(c, "shrout")
    COLS_TIPO = [x for x in ["securitytype", "securitysubtype", "sharetype"] if x in c]
    BAJAS_INCLUIDAS = True      # en el formato nuevo la rentabilidad de salida ya va incluida
else:
    MSF, COL_FECHA, COL_RET, COL_PRC, COL_SHR = "msf", "date", "ret", "prc", "shrout"
    COLS_TIPO, BAJAS_INCLUIDAS = [], False
print(f"Rentabilidades: crsp.{MSF} ({COL_FECHA}, {COL_RET}, {COL_PRC}, {COL_SHR})")

# Enlace CRSP-Compustat
link = None
for tabla in ["crsp.ccmxpf_lnkhist", "crsp_a_ccm.ccmxpf_lnkhist"]:
    link = intentar(f'''
        select gvkey, lpermno as permno, linkdt, linkenddt from {tabla}
        where linktype in ('LU', 'LC') and linkprim in ('P', 'C')
    ''', ["linkdt", "linkenddt"], tabla)
    if link is not None:
        print(f"Enlace CRSP-Compustat: {tabla}")
        break
if link is None:
    raise RuntimeError("Sin acceso al enlace CRSP-Compustat. Hay que preguntar a la biblioteca de CUNEF.")
link = link.dropna(subset=["permno"]).copy()
link["permno"] = link.permno.astype(int)
link["linkenddt"] = link.linkenddt.fillna(pd.Timestamp("2099-12-31"))

## 4. El universo de empresas y los datos
**Qué empresas entran cada año.** Lo ideal es la composición histórica oficial del S&P 500. Esta celda la busca en CRSP y en Compustat. Si la suscripción de CUNEF no la incluye, construye un universo equivalente: **cada mes, las 500 mayores empresas de EE. UU. por capitalización** (solo acciones ordinarias con ficha en Compustat, sin fondos). Coincide casi del todo con el S&P 500, también evita el sesgo de supervivencia y es un método habitual en estudios académicos.

Después descarga las rentabilidades, los fundamentales y los sectores. Puede tardar unos minutos.

In [ ]:
# 4.1 Composición histórica del S&P 500, si hay acceso
sp, FUENTE_UNIVERSO = None, None
for tabla in ["dsp500list_v2", "dsp500list"]:
    if sp is None and tabla in tablas_crsp:
        try:
            c = columnas("crsp", tabla)
            c_ini, c_fin = elegir(c, "mbrstartdt", "start"), elegir(c, "mbrenddt", "ending")
            sp = intentar(f"select permno, {c_ini} as ini, {c_fin} as fin from crsp.{tabla} "
                          f"where {c_fin} >= '{INICIO_DATOS}'", ["ini", "fin"], f"crsp.{tabla}")
            if sp is not None:
                FUENTE_UNIVERSO = f"S&P 500 histórico (crsp.{tabla})"
        except Exception as e:
            print(f"   · crsp.{tabla}: no disponible ({str(e).splitlines()[0][:100]})")
if sp is None:
    idx = intentar(f'''
        select gvkey, "from" as ini, thru as fin from comp.idxcst_his
        where gvkeyx = '000003' and (thru is null or thru >= '{INICIO_DATOS}')
    ''', ["ini", "fin"], "comp.idxcst_his")
    if idx is not None and len(idx):
        sp = idx.merge(link[["gvkey", "permno"]].drop_duplicates(), on="gvkey")[["permno", "ini", "fin"]]
        FUENTE_UNIVERSO = "S&P 500 histórico (comp.idxcst_his)"

# Comprobar que la lista histórica está completa: el S&P 500 tiene ~500 acciones todos los años.
# Si faltan las empresas que salieron del índice, usarla introduciría sesgo de supervivencia.
if sp is not None:
    sp["fin"] = sp.fin.fillna(pd.Timestamp("2099-12-31"))
    conteos = {a: sp[(sp.ini <= pd.Timestamp(f"{a}-01-01")) &
                     (sp.fin >= pd.Timestamp(f"{a}-01-01"))].permno.nunique() for a in range(2015, 2026)}
    print(f"Acciones por año en {FUENTE_UNIVERSO}: {conteos}")
    completa = min(conteos.values()) >= 0.94 * TAMANO_UNIVERSO and max(conteos.values()) <= 1.08 * TAMANO_UNIVERSO
    if UNIVERSO == "mayores" or not completa:
        if not completa:
            print("   · La lista está INCOMPLETA (debería haber ~500 cada año): le faltan empresas que salieron "
                  "del índice. Para evitar el sesgo de supervivencia, uso las mayores empresas de CRSP.")
        sp, FUENTE_UNIVERSO = None, None

USA_SP500 = sp is not None
if USA_SP500:
    sp["permno"] = sp.permno.astype(int)
    sp["fin"] = sp.fin.fillna(pd.Timestamp("2099-12-31"))
    NOMBRE_INDICE = "S&P 500"
else:
    FUENTE_UNIVERSO = f"las {TAMANO_UNIVERSO} mayores empresas de EE. UU. por capitalización (CRSP)"
    NOMBRE_INDICE = f"{TAMANO_UNIVERSO} mayores de EE. UU."
print(f"\nUniverso utilizado: {FUENTE_UNIVERSO}\n")

# 4.2 Rentabilidades mensuales (con dividendos) y capitalización
filtro = f"and permno in {sql_lista(sp.permno.unique())}" if USA_SP500 else ""
extra = "".join(f", {x}" for x in COLS_TIPO)
msf = db.raw_sql(f'''
    select permno, {COL_FECHA} as fecha, {COL_RET} as ret, {COL_PRC} as prc, {COL_SHR} as shrout{extra}
    from crsp.{MSF}
    where {COL_FECHA} between '{INICIO_DATOS}' and '{FIN_DATOS}' {filtro}
''', date_cols=["fecha"])
msf["permno"] = msf.permno.astype(int)
msf["mes"] = msf.fecha.dt.to_period("M")

if not BAJAS_INCLUIDAS:
    # Formato antiguo: añadir la rentabilidad del día en que la acción deja de cotizar
    bajas = intentar(f"select permno, dlstdt, dlret from crsp.msedelist where dlstdt >= '{INICIO_DATOS}' {filtro}",
                     ["dlstdt"], "crsp.msedelist")
    if bajas is not None:
        bajas["permno"] = bajas.permno.astype(int)
        bajas["mes"] = bajas.dlstdt.dt.to_period("M")
        msf = msf.merge(bajas[["permno", "mes", "dlret"]], on=["permno", "mes"], how="left")
        msf["ret"] = np.where(msf.dlret.notna(), (1 + msf.ret.fillna(0)) * (1 + msf.dlret) - 1, msf.ret)
        msf = msf.drop(columns=["dlret"])

msf["mcap"] = msf.prc.abs() * msf.shrout

# 4.3 Universo de las mayores empresas (solo si no hay lista histórica del S&P 500)
if not USA_SP500:
    elegibles = msf[msf.permno.isin(set(link.permno))]          # con ficha en Compustat: excluye fondos
    if COLS_TIPO:
        comun = pd.Series(True, index=elegibles.index)
        if "securitytype" in elegibles:
            comun &= elegibles.securitytype.eq("EQTY")
        if "securitysubtype" in elegibles:
            comun &= elegibles.securitysubtype.eq("COM")
        if "sharetype" in elegibles:
            comun &= elegibles.sharetype.eq("NS")             # excluye ADR y otros tipos
        if comun.mean() > 0.2:
            elegibles = elegibles[comun]
        else:
            print("   · aviso: el filtro de acciones ordinarias descartaría casi todo; no se aplica")
    else:
        nombres = intentar("select distinct permno from crsp.msenames where shrcd in (10, 11, 12)",
                           aviso="crsp.msenames")
        if nombres is not None:
            elegibles = elegibles[elegibles.permno.isin(set(nombres.permno.astype(int)))]
    top = (elegibles.dropna(subset=["mcap"])
                    .sort_values(["mes", "mcap"], ascending=[True, False])
                    .groupby("mes").head(TAMANO_UNIVERSO + 100))     # candidatos; se depuran en 4.6
    MIEMBROS_POR_MES = top.groupby("mes").permno.apply(lambda x: set(x.astype(int))).to_dict()


def miembros_en(fecha):
    # Empresas del universo en una fecha (con la capitalización del mes anterior, sin mirar al futuro)
    if USA_SP500:
        return sp[(sp.ini <= fecha) & (sp.fin >= fecha)].permno.unique()
    return list(MIEMBROS_POR_MES.get((fecha - pd.Timedelta(days=1)).to_period("M"), set()))


permnos_universo = set(sp.permno) if USA_SP500 else set().union(*MIEMBROS_POR_MES.values())
link = link[link.permno.isin(permnos_universo)].copy()
lista_gvkeys = sql_lista(link.gvkey.unique())
print(f"Empresas que han pasado por el universo desde 2012: {link.gvkey.nunique()}")

# 4.4 Fundamentales trimestrales (en millones de $) con fecha de publicación
fq = db.raw_sql(f'''
    select gvkey, datadate, rdq, curcdq, oiadpq, actq, lctq, cheq, dlcq, dlttq, ppentq, pstkq, mibq, intanq
    from comp.fundq
    where indfmt = 'INDL' and datafmt = 'STD' and popsrc = 'D' and consol = 'C'
      and datadate between '{INICIO_DATOS}' and '{FIN_DATOS}' and gvkey in {lista_gvkeys}
''', date_cols=["datadate", "rdq"])
fq = fq.sort_values(["gvkey", "datadate"]).drop_duplicates(["gvkey", "datadate"], keep="last")
# Si falta la fecha de publicación, asumimos 90 días después del cierre del trimestre (prudente)
fq["disponible"] = fq.rdq.fillna(fq.datadate + pd.Timedelta(days=90))

# 4.5 Nombre y sector de cada empresa
comp = db.raw_sql(f"select gvkey, conm, gsector, sic from comp.company where gvkey in {lista_gvkeys}")

# 4.6 Solo empresas que presentan sus cuentas en dólares. Algunas extranjeras cotizan en EE. UU.
#     (p. ej. CGI en dólares canadienses o ASML en euros): mezclar su EBIT en otra divisa con su
#     capitalización en dólares falsearía el Earnings Yield.
moneda = fq.dropna(subset=["curcdq"]).groupby("gvkey").curcdq.agg(lambda x: x.mode().iat[0])
gvkeys_no_usd = set(moneda[moneda != "USD"].index)
permnos_no_usd = set(link[link.gvkey.isin(gvkeys_no_usd)].permno)
if not USA_SP500:
    top = top[~top.permno.isin(permnos_no_usd)].groupby("mes").head(TAMANO_UNIVERSO)
    MIEMBROS_POR_MES = top.groupby("mes").permno.apply(lambda x: set(x.astype(int))).to_dict()
print(f"Excluidas por presentar cuentas en otra divisa: {len(gvkeys_no_usd)} empresas "
      f"(p. ej. {', '.join(comp[comp.gvkey.isin(gvkeys_no_usd)].conm.head(5))})")

print(f"Rentabilidades: {len(msf):,} filas | Fundamentales: {len(fq):,} trimestres | Empresas: {len(comp):,}")

## 5. Comprobaciones de calidad
Antes de calcular nada, compruebo que los datos tienen sentido. Un estudio profesional siempre hace esto.

In [ ]:
# Unidades de la capitalización: las 500 mayores de EE. UU. valen entre 15 y 60 billones de $ en este periodo
dentro = miembros_en(pd.Timestamp("2025-01-01"))
ref = pd.Period("2024-12", "M")
total = msf[(msf.mes == ref) & msf.permno.isin(dentro)].mcap.sum()
msf["mcap"] = msf.mcap / (1e6 if total > 1e12 else 1e3)        # pasar a millones de $
total_m = msf[(msf.mes == ref) & msf.permno.isin(dentro)].mcap.sum()
print(f"Capitalización total del universo ({NOMBRE_INDICE}) en dic-2024: "
      f"{total_m / 1e6:,.1f} billones de $  (esperado: ~45-55)")

# Último mes con datos completos
cobertura = msf.dropna(subset=["ret"]).groupby("mes").permno.nunique()
ULTIMO_MES = cobertura[cobertura >= 0.8 * cobertura.max()].index.max()
print(f"Último mes con datos de rentabilidad: {ULTIMO_MES}")

for a in [2015, 2020, 2025]:
    print(f"Empresas en el universo el 1/1/{a}: {len(miembros_en(pd.Timestamp(f'{a}-01-01')))}")

## 6. El ranking de la Magic Formula en una fecha (sin mirar al futuro)
Para cada fecha:
1. Empresas que estaban en el universo ese día y su capitalización a cierre del mes anterior.
2. Para cada empresa, los **4 últimos trimestres publicados antes de esa fecha**: EBIT de 12 meses (`oiadpq`) y el último balance.
3. Se excluyen financieras, utilities e inmobiliarias (sectores GICS 40, 55 y 60).
4. **EV** = capitalización + deuda (corto y largo plazo) + preferentes + minoritarios − caja.
5. **Capital invertido** = capital circulante neto (mínimo cero) + inmovilizado material neto (+ intangibles en la variante `gw`).
6. Se ordena por Earnings Yield y por ROC, se suman los dos puestos y gana la suma más baja.

In [ ]:
CAMPOS_BALANCE = ["actq", "lctq", "cheq", "dlcq", "dlttq", "ppentq", "pstkq", "mibq", "intanq"]

def foto_fundamental(fecha):
    # Últimos datos publicados antes de 'fecha' para cada empresa
    d = fq[(fq.disponible < fecha) & (fq.datadate >= fecha - pd.Timedelta(days=550))
           & (fq.curcdq.fillna("USD") == "USD")]
    filas = []
    for gvkey, g in d.groupby("gvkey"):
        g = g.tail(4)
        if len(g) < 4 or g.oiadpq.isna().any():
            continue                                         # sin 4 trimestres completos
        if (g.datadate.iloc[-1] - g.datadate.iloc[0]).days > 300:
            continue                                         # trimestres no consecutivos
        ult = g.iloc[-1]
        if (fecha - ult.datadate).days > 270:
            continue                                         # balance demasiado antiguo
        filas.append({"gvkey": gvkey, "ebit": g.oiadpq.sum(), "ultimo_trimestre": ult.datadate,
                      **{k: ult[k] for k in CAMPOS_BALANCE}})
    return pd.DataFrame(filas)


def ranking(fecha, roc_metodo=ROC_METODO):
    miembros = miembros_en(fecha)
    mes_anterior = (fecha - pd.Timedelta(days=1)).to_period("M")
    cap = msf[(msf.mes == mes_anterior) & msf.permno.isin(miembros)][["permno", "mcap"]].dropna()
    enlaces = link[(link.linkdt <= fecha) & (link.linkenddt >= fecha)][["gvkey", "permno"]]
    cap = cap.merge(enlaces, on="permno")
    # Empresas con dos clases de acciones (p. ej. Alphabet): sumar capitalización, quedarse con la mayor
    cap = (cap.sort_values("mcap", ascending=False)
              .groupby("gvkey").agg(permno=("permno", "first"), mcap=("mcap", "sum")).reset_index())

    df = cap.merge(foto_fundamental(fecha), on="gvkey").merge(comp, on="gvkey", how="left")

    sic = pd.to_numeric(df.sic, errors="coerce")
    excluida = df.gsector.isin(["40", "55", "60"]) | (
        df.gsector.isna() & (sic.between(6000, 6999) | sic.between(4900, 4999)))
    df = df[~excluida].copy()

    z = lambda s: s.fillna(0)
    df["ev"] = df.mcap + z(df.dlttq) + z(df.dlcq) + z(df.pstkq) + z(df.mibq) - z(df.cheq)
    df["capital_circulante"] = (df.actq - z(df.cheq)) - (df.lctq - z(df.dlcq))
    df["capital"] = df.capital_circulante.clip(lower=0).fillna(0) + z(df.ppentq)
    if roc_metodo == "gw":
        df["capital"] = df.capital + z(df.intanq)
    df["ey"] = df.ebit / df.ev
    df["roc"] = df.ebit / df.capital
    df = df[(df.ebit > 0) & (df.ev > 0) & (df.capital > 0)].copy()

    df["rank_ey"] = df.ey.rank(ascending=False, method="first")
    df["rank_roc"] = df.roc.rank(ascending=False, method="first")
    df["puntos"] = df.rank_ey + df.rank_roc
    df = df.sort_values(["puntos", "ey"], ascending=[True, False]).reset_index(drop=True)
    df["puesto"] = df.index + 1
    return df

### Prueba: el ranking del 1 de enero de 2025
Revisa que las empresas y las cifras tengan sentido (EY y ROC en tanto por uno: 0,10 = 10 %).

In [ ]:
prueba = ranking(pd.Timestamp("2025-01-01"))
print(f"Empresas que pasan los filtros: {len(prueba)}")
prueba[["puesto", "conm", "mcap", "ebit", "ev", "ey", "roc", "ultimo_trimestre"]].head(15)

## 7. Rentabilidad de la cartera y de los índices
- **Cartera**: mismo importe en cada empresa el 1 de enero y se mantiene todo el año (sin rebalancear). Si una empresa deja de cotizar, su dinero queda en liquidez.
- **S&P 500 ponderado**: cada mes, los miembros ponderados por su capitalización del mes anterior (equivale al índice con dividendos).
- **S&P 500 a partes iguales**: los mismos miembros, con el mismo peso. Es la comparación más justa, porque la cartera también pondera igual.

In [ ]:
def meses_del_anio(anio):
    return [m for m in pd.period_range(f"{anio}-01", f"{anio}-12", freq="M") if m <= ULTIMO_MES]


def rentabilidad_cartera(permnos, anio):
    meses = meses_del_anio(anio)
    r = (msf[msf.permno.isin(permnos) & msf.mes.isin(meses)]
         .pivot_table(index="mes", columns="permno", values="ret")
         .reindex(index=meses, columns=permnos))
    valor = (1 + r.fillna(0)).cumprod()          # valor de cada posición (sin datos = liquidez)
    cartera = valor.mean(axis=1)                 # mismo peso inicial
    return cartera / cartera.shift(1, fill_value=1.0) - 1


def rentabilidad_indice(anio, ponderado=True):
    salida = {}
    for m in meses_del_anio(anio):
        miembros = miembros_en(m.start_time)
        previo = msf[(msf.mes == m - 1) & msf.permno.isin(miembros)][["permno", "mcap"]]
        actual = msf[(msf.mes == m) & msf.permno.isin(miembros)][["permno", "ret"]]
        x = previo.merge(actual, on="permno").dropna()
        salida[m] = np.average(x.ret, weights=x.mcap) if ponderado else x.ret.mean()
    return pd.Series(salida)

## 8. El backtest completo

In [ ]:
def backtest(top_n=TOP_N, roc_metodo=ROC_METODO, verbose=True):
    mensual, carteras = [], []
    for anio in ANIOS:
        fecha = pd.Timestamp(f"{anio}-01-01")
        if not meses_del_anio(anio):
            break
        rk = ranking(fecha, roc_metodo)
        top = rk.head(top_n)
        carteras.append(top.assign(anio=anio)[["anio", "puesto", "conm", "ey", "roc"]])
        tabla = pd.DataFrame({
            "estrategia": rentabilidad_cartera(list(top.permno), anio),
            "sp500": rentabilidad_indice(anio, ponderado=True),
            "sp500_igual_peso": rentabilidad_indice(anio, ponderado=False),
        })
        mensual.append(tabla)
        if verbose:
            r = (1 + tabla).prod() - 1
            print(f"{anio}: estrategia {r.estrategia:+7.1%} | {NOMBRE_INDICE} {r.sp500:+7.1%} | "
                  f"igual peso {r.sp500_igual_peso:+7.1%} | {len(rk)} candidatas")
    return pd.concat(mensual), pd.concat(carteras)


mensual, carteras = backtest()

## 9. Resultados
Métricas habituales:
- **Rentabilidad anual compuesta (CAGR)**: lo que habrías ganado de media cada año.
- **Volatilidad**: cuánto sube y baja (desviación típica anualizada).
- **Máxima caída**: la peor bajada desde un máximo, lo que más "duele" aguantar.
- **Años por encima del índice**: cuántos años gana la estrategia.

In [ ]:
def metricas(serie):
    años = len(serie) / 12
    curva = (1 + serie).cumprod()
    return pd.Series({
        "Rentabilidad total": curva.iloc[-1] - 1,
        "Rentabilidad anual (CAGR)": curva.iloc[-1] ** (1 / años) - 1,
        "Volatilidad anual": serie.std() * np.sqrt(12),
        "Máxima caída": (curva / curva.cummax() - 1).min(),
    })

anual = mensual.groupby(mensual.index.year).apply(lambda x: (1 + x).prod() - 1)
anual["diferencia_vs_sp500"] = anual.estrategia - anual.sp500

resumen = mensual.apply(metricas).T
resumen[f"Años por encima de {NOMBRE_INDICE}"] = [
    f"{(anual.estrategia > anual.sp500).sum()} de {len(anual)}", "—",
    f"{(anual.estrategia > anual.sp500_igual_peso).sum()} de {len(anual)} (vs igual peso)"]

print("Rentabilidad por año")
display(anual.style.format("{:+.1%}"))
print("Resumen del periodo")
display(resumen.style.format({c: "{:.1%}" for c in resumen.columns[:4]}))

In [ ]:
curvas = (1 + mensual).cumprod() * 10_000
ax = curvas.rename(columns={"estrategia": f"Magic Formula (top {TOP_N})", "sp500": NOMBRE_INDICE,
                            "sp500_igual_peso": f"{NOMBRE_INDICE}, igual peso"}).plot(figsize=(11, 5.5), lw=2)
ax.set_title("Evolución de 10.000 $ invertidos el 1 de enero de 2015 (con dividendos)")
ax.set_ylabel("Valor de la inversión ($)")
ax.set_xlabel("")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Variantes
¿Cambia mucho el resultado con otras decisiones? Si una estrategia solo funciona con unos parámetros muy concretos, no es fiable.

In [ ]:
variantes = {
    "Top 30 · ROC Greenblatt": dict(top_n=30, roc_metodo="tangible"),
    "Top 50 · ROC Greenblatt": dict(top_n=50, roc_metodo="tangible"),
    "Top 30 · ROC con fondo de comercio": dict(top_n=30, roc_metodo="gw"),
}
comparacion = {}
for nombre, params in variantes.items():
    m, _ = backtest(verbose=False, **params)
    comparacion[nombre] = metricas(m.estrategia)
comparacion[NOMBRE_INDICE] = metricas(mensual.sp500)
comparacion[f"{NOMBRE_INDICE}, igual peso"] = metricas(mensual.sp500_igual_peso)
display(pd.DataFrame(comparacion).T.style.format("{:.1%}"))

## 11. Guardar los resultados agregados
Se guardan en la carpeta `resultados_backtest` de tu espacio de JupyterHub. Desde el panel de archivos de la izquierda puedes descargarlos (botón derecho → *Download*).

> Solo contienen resultados del estudio (rentabilidades de la cartera y de los índices, y las empresas elegidas cada año). Antes de publicarlos en una web, confirma con la biblioteca de CUNEF que está permitido.

In [ ]:
import os
os.makedirs("resultados_backtest", exist_ok=True)
mensual.rename_axis("mes").to_csv("resultados_backtest/rentabilidad_mensual.csv")
anual.rename_axis("anio").to_csv("resultados_backtest/rentabilidad_anual.csv")
resumen.to_csv("resultados_backtest/resumen.csv")
carteras.to_csv("resultados_backtest/carteras_por_anio.csv", index=False)
print("Guardado en resultados_backtest/:", os.listdir("resultados_backtest"))